# NumPy 신경망으로 손글씨 숫자 인식

이 노트북은 과제 문서 「과제 - 신경망을 이용한 손글씨 숫자 인식」의 Day 1~3 순서를 따릅니다. 목표는 정확도 경쟁이 아니라 `Forward -> Loss -> Backward -> Update`를 직접 구현하고 설명하는 것입니다.

- 현재 Day의 TODO와 테스트를 끝낸 뒤 다음으로 이동합니다.
- 작은 배열 검산과 gradient check를 전체 학습보다 먼저 수행합니다.
- learning-rate·optimizer·hidden-size 비교는 선택입니다.

## 0. 환경설정 - 제공 코드

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, subprocess, getpass
from pathlib import Path

PROJECT_DIR = 'mnist-lab'
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not Path(PROJECT_DIR).exists():
    url = input('저장소 URL: ').strip().replace('https://', '').replace('http://', '')
    token = getpass.getpass('GitHub token (공개 저장소면 Enter): ').strip()
    remote = f'https://{token}@{url}' if token else f'https://{url}'
    subprocess.run(['git', 'clone', remote, PROJECT_DIR], check=True)

ROOT = (Path(PROJECT_DIR) if Path(PROJECT_DIR).exists() else Path('.')).resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('ROOT =', ROOT)

In [ ]:
import numpy as np

from activations import ReLU
from data import load_mnist
from layers import Affine
from losses import SoftmaxWithLoss
from network import TwoLayerNet
from optimizers import SGD, Adam
from training import evaluate, gradient_check, plot_loss_history, train

(x_train, y_train), (x_test, y_test) = load_mnist()
print('train:', x_train.shape, y_train.shape)
print('test :', x_test.shape, y_test.shape)
print('pixel range:', x_train.min(), '~', x_train.max())

In [ ]:
def run_pytest(targets):
    command = [sys.executable, '-m', 'pytest', *targets.split(), '-q', '--no-header']
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout[-4000:])
    if result.stderr:
        print(result.stderr[-2000:])
    result.check_returncode()

def selfcheck(day):
    result = subprocess.run(
        [sys.executable, 'tools/selfcheck.py', '--day', str(day)],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

## Day 1 - Forward, logits, loss

구현: `Affine.forward`, `ReLU.forward`, `SoftmaxWithLoss.forward`. 손으로 계산 가능한 값부터 확인합니다.

In [ ]:
run_pytest(
    'tests/test_affine.py::TestAffineForward '
    'tests/test_relu.py::TestReLUForward '
    'tests/test_softmax_with_loss.py::TestSoftmaxWithLossForward'
)

### 1-1. Affine 값과 bias broadcasting

In [ ]:
W = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])
b = np.array([1.0, 0.0, -1.0])
x = np.array([[1.0, 2.0],
              [3.0, 4.0]])
out = Affine(W, b).forward(x)
print('x', x.shape, '@ W', W.shape, '+ b', b.shape, '->', out.shape)
print(out)
print('기대값: [[10, 12, 14], [20, 26, 32]]')

### 1-2. ReLU와 network shape

In [ ]:
sample = np.array([[-2.0, -0.5, 0.0, 0.5, 2.0]])
print('ReLU input :', sample)
print('ReLU output:', ReLU().forward(sample))

model = TwoLayerNet(hidden_size=128, seed=0)
value = x_train[:4]
print('input         ', value.shape)
for layer in model.layers:
    value = layer.forward(value)
    print(f'{type(layer).__name__:14s}', value.shape)
print('마지막 값은 확률이 아니라 logits입니다.')

### 1-3. 초기 loss와 `ln(10)`

모든 logits가 같으면 loss는 정확히 `ln(10) = 2.3026`입니다. 실제 모델은 He 초기화로 logits가 조금씩 다르므로 초기 loss가 `ln(10)`과 완전히 같지 않고 2.3 근처(대략 2.0~2.8)에 있으면 정상입니다. 이 범위를 크게 벗어나면 `SoftmaxWithLoss.forward`를 먼저 확인하세요.

In [ ]:
model = TwoLayerNet(seed=0)
criterion = SoftmaxWithLoss()
logits = model.forward(x_train[:128])
loss = criterion.forward(logits, y_train[:128])
print(f'initial loss: {loss:.4f}')
print(f'ln(10)      : {np.log(10):.4f}')
print('첫 sample 확률 합:', criterion.y_prob[0].sum())

In [ ]:
selfcheck(1)

## Day 2 - Backward와 chain rule

구현 순서: `SoftmaxWithLoss.backward` -> `ReLU.backward` -> `Affine.backward`. 마지막에 수치 미분과 비교합니다.

In [ ]:
run_pytest(
    'tests/test_softmax_with_loss.py '
    'tests/test_relu.py '
    'tests/test_affine.py '
    'tests/test_gradient_check.py'
)

### 2-1. Affine backward shape

In [ ]:
layer = Affine(np.random.randn(784, 128), np.zeros(128))
batch = x_train[:4]
layer.forward(batch)
dx = layer.backward(np.random.randn(4, 128))
print('dx:', dx.shape, '== x:', batch.shape)
print('dW:', layer.dW.shape, '== W:', layer.W.shape)
print('db:', layer.db.shape, '== b:', layer.b.shape)

### 2-2. Gradient check

수치 미분과 backward 결과를 비교합니다. W2/b2 실패는 출력층과 두 번째 Affine, W1/b1 실패는 ReLU와 첫 번째 Affine부터 확인합니다.

In [ ]:
check_model = TwoLayerNet(hidden_size=16, seed=0)
ok, max_error = gradient_check(
    check_model, SoftmaxWithLoss(), x_train[:8], y_train[:8], num_checks=6
)
print('통과:', ok, '| 최대 상대 오차:', max_error)

In [ ]:
selfcheck(2)

## Day 3 - SGD와 end-to-end 학습

구현: `SGD.update`, `train()`의 Forward, Loss, Backward, Update. 먼저 인공 데이터 테스트를 통과한 뒤 MNIST baseline을 한 번 실행합니다.

In [ ]:
run_pytest('tests/test_sgd.py tests/test_training_smoke.py')

### 3-1. Synthetic Smoke

MNIST 다운로드 없이 학습 흐름을 확인하는 제공 코드입니다. 첫·마지막 loss와 parameter 변화를 `REPORT.md`에 기록하세요.

In [ ]:
def make_synthetic_smoke_data(n_samples=1000, seed=0):
    rng = np.random.default_rng(seed)
    patterns = np.zeros((10, 784))
    for class_id in range(10):
        on_pixels = rng.choice(784, size=40, replace=False)
        patterns[class_id, on_pixels] = 1.0
    y = rng.integers(0, 10, n_samples)
    x = np.clip(patterns[y] + rng.normal(0.0, 0.2, (n_samples, 784)), 0.0, 1.0)
    return x, y

x_smoke, y_smoke = make_synthetic_smoke_data()
smoke_model = TwoLayerNet(seed=0)
before = {key: value.copy() for key, value in smoke_model.params.items()}
smoke_history = train(
    smoke_model, SoftmaxWithLoss(), Adam(lr=0.001),
    x_smoke, y_smoke, epochs=5, batch_size=128, verbose=False
)
params_changed = all(
    not np.allclose(smoke_model.params[key], value) for key, value in before.items()
)
print(f'loss: {smoke_history[0]:.3f} -> {smoke_history[-1]:.3f}')
print('all parameters changed:', params_changed)

### 3-2. MNIST baseline 1회

정확도는 관찰값이며 통과 기준이 아닙니다. 설정을 반복 변경하지 말고 loss 감소와 학습 전후 예측 변화를 확인합니다.

In [ ]:
model = TwoLayerNet(hidden_size=128, seed=0)
history = train(
    model, SoftmaxWithLoss(), SGD(lr=0.1),
    x_train, y_train, epochs=10, batch_size=128
)
accuracy, n_params = evaluate(model, x_test, y_test)
print(f'loss: {history[0]:.4f} -> {history[-1]:.4f}')
print(f'observed accuracy: {accuracy:.2f}%')
print(f'parameters: {n_params:,}')
plot_loss_history(history, label='baseline')

In [ ]:
untrained = TwoLayerNet(seed=0)
print('target        :', y_test[:10])
print('before train  :', untrained.predict(x_test[:10]).argmax(axis=1))
print('after train   :', model.predict(x_test[:10]).argmax(axis=1))

In [ ]:
run_pytest('tests/')
selfcheck(3)

### 제출 전 확인

- 전체 테스트와 gradient check 결과를 기록했는가?
- `dx`, `dW`, `db`의 shape와 식을 설명할 수 있는가?
- loss 감소를 정확도 경쟁이 아니라 연결 검증으로 설명했는가?
- MNIST baseline을 한 번 실행하고 조건과 결과를 기록했는가?
- AI·교재·동료의 도움과 직접 검증 방법을 기록했는가?

## 선택 심화

필수 작업을 모두 마친 뒤에만 learning rate 하나, SGD/Adam, BatchNorm/Dropout 중 한 항목을 선택해 관찰할 수 있습니다. 선택 결과는 평가용 성능 순위가 아닙니다.